# EO4FLOOD snow products

Snow cover and stored snow water describe different parts of a catchment's seasonal water storage.
Snow-covered area can change without a proportional change in water equivalent, and neither
quantity alone determines meltwater runoff. This demonstration reads three online Zarr assets
identified by the EO4FLOOD STAC items over a small Torne-area window during March–May 2020.

| Example product | Variable | Interpretation |
|---|---|---|
| ESA CCI AVHRR METOP-A | `scfg` | Snow cover fraction on ground, percent |
| ESA CCI SWE | `swe` | Snow water equivalent, mm of water |
| CLMS Northern Hemisphere SCE | `sce` | Delivered snow cover extent, percent |

The first two products are illustrated through a spring season. The finer SCE product is shown
for a single date. The examples retain their own grids; percentages and water depth are not
interchangeable measurements.

## Snow volume and decompression

Compressed size can greatly understate the memory needed to read a snow product. The high-resolution
FSC and GFSC deliveries contain both `snow_mask` and `nodata_mask` across a large spatial grid.
Their shapes and chunk sizes below are metadata snapshots from the selected STAC assets; their
pixel arrays are not opened or read in this demonstration.

In those delivered binary masks, `snow_mask == 1` identifies snow, while zero can mean either
snow-free or unobserved. `nodata_mask == 1` must be excluded before a snow-covered fraction is
calculated. These masks are not continuous fractional-cover percentages.

The data examples instead use native chunks below 32 MiB, select a small geographic and temporal
window before computing, and execute one chunk task at a time. A small output alone does not make
a read cheap: the complete native chunk still has to decompress. The reported chunk sizes are raw
array sizes, not a promise about total process memory, which also includes temporary buffers.

In [ ]:
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import xarray as xr
import s3fs
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True})
mask_catalogue = pd.DataFrame([{'product': 'FSC', 'shape': [366, 14667, 27333], 'native_chunks': [10, 2934, 5467], 'dtype': '|u1', 'mask_arrays': 2}, {'product': 'GFSC', 'shape': [366, 4583, 8542], 'native_chunks': [10, 2292, 4271], 'dtype': '|u1', 'mask_arrays': 2}])
mask_catalogue["raw_two_mask_GiB"] = mask_catalogue.apply(
    lambda row: np.prod(row["shape"]) * np.dtype(row["dtype"]).itemsize * row.mask_arrays / 2**30, axis=1)
mask_catalogue["raw_chunk_MiB"] = mask_catalogue.apply(
    lambda row: np.prod(row.native_chunks) * np.dtype(row["dtype"]).itemsize / 2**20, axis=1)
display(mask_catalogue)

In [ ]:
s3 = s3fs.S3FileSystem(anon=True, client_kwargs={
    "endpoint_url": "https://s3.waw4-1.cloudferro.com"
})
PRODUCT_URLS = {
    "SCFG": "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/eo4flood/Snow_zarr/EO4FLOOD_Snow_CCI_SCFG_AVHRR_METOPA_tore_20160101_20211231.zarr",
    "SWE": "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/eo4flood/Snow_zarr/EO4FLOOD_Snow_CCI_SWE_tore_20160101_20231231.zarr",
    "SCE": "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/eo4flood/Snow_zarr/EO4FLOOD_Snow_CLMS_SCE_NH_tore_20180109_20241231.zarr"
}
datasets = {}
for product, url in PRODUCT_URLS.items():
    datasets[product] = xr.open_zarr(
        s3.get_mapper(urlparse(url).path.lstrip("/")), consolidated=False, chunks={})
fields = {"SCFG": datasets["SCFG"].scfg, "SWE": datasets["SWE"].swe, "SCE": datasets["SCE"].sce}
chunk_sizes = pd.Series({
    name: np.prod(field.encoding["chunks"]) * np.dtype(field.encoding["dtype"]).itemsize / 2**20
    for name, field in fields.items()
}, name="Native chunk MiB")
assert chunk_sizes.max() <= 32, "The selected source chunk exceeds the demonstration's decompression budget."
display(chunk_sizes.to_frame())

## A bounded spring sample

The geographic window spans 67.2–68.2° N and 22–23° E. The latitude coordinate runs south to north.
The seasonal samples cover 1 March through 31 May 2020; the SCE map is restricted to 15 May.
These are regional samples rather than whole-basin statistics.

Only the selected fields are computed. The selected outputs are each limited to 8 MiB and the
Dask scheduler executes tasks synchronously, avoiding concurrent decompression of many chunks.
No full-store load, persistence, or high-resolution mask scan is performed.

In [ ]:
selections = {
    "SCFG": fields["SCFG"].sel(time=slice("2020-03-01", "2020-05-31"),
                              lat=slice(67.2, 68.2), lon=slice(22, 23)),
    "SWE": fields["SWE"].sel(time=slice("2020-03-01", "2020-05-31"),
                            lat=slice(67.2, 68.2), lon=slice(22, 23)),
    "SCE": fields["SCE"].sel(time="2020-05-15", lat=slice(67.2, 68.2), lon=slice(22, 23)),
}
selection_sizes = pd.Series({name: field.nbytes / 2**20 for name, field in selections.items()},
                            name="Selected output MiB")
assert all(field.size > 0 for field in selections.values())
assert selection_sizes.max() <= 8, "The selected output exceeds the demonstration's memory budget."
display(selection_sizes.to_frame())
samples = {}
for name, field in selections.items():
    samples[name] = field.compute(scheduler="synchronous")
scfg = samples["SCFG"].where((samples["SCFG"] >= 0) & (samples["SCFG"] <= 100))
swe = samples["SWE"].where(samples["SWE"] >= 0)
sce = samples["SCE"].where((samples["SCE"] >= 100) & (samples["SCE"] <= 200)) - 100

## Ground snow fraction during spring

Three fixed dates show how the fraction of ground covered by snow varies across the sample.
The scale is held at 0–100% across panels. Missing values remain blank, and values outside the
physical percentage range are excluded. The observed ground fraction is not a snow-depth estimate.
The April example also illustrates that a selected date can have no valid observations in this window.

In [ ]:
MAP_DATES = ["2020-03-15", "2020-04-15", "2020-05-15"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for date, ax in zip(MAP_DATES, axes):
    scfg.sel(time=date).plot(ax=ax, x="lon", y="lat", cmap="Blues", vmin=0, vmax=100,
                             cbar_kwargs={"label": "Ground snow fraction (%)"})
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.set_title(date)
fig.tight_layout()
plt.show()

## Snow water equivalent

Snow water equivalent expresses the water contained in the snowpack as a depth of liquid water.
It describes storage rather than the fraction of land covered. The three maps use a shared colour
scale derived only from the small computed sample. Negative water-equivalent values are excluded.
A change in SWE is not interpreted directly as runoff: snowfall, melt, redistribution and other
processes can contribute to the observed change.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
maximum_swe = float(swe.max())
for date, ax in zip(MAP_DATES, axes):
    swe.sel(time=date).plot(ax=ax, x="lon", y="lat", cmap="Blues", vmin=0, vmax=maximum_swe,
                            cbar_kwargs={"label": "Snow water equivalent (mm)"})
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.set_title(date)
fig.tight_layout()
plt.show()

## Spring evolution and observed coverage

Spatial means use valid cells within each product's selected window. The products have different
grid spacing and observation characteristics, so their means are not a pixel-level comparison.
The coverage panel shows the fraction of selected cells with valid data on each day; missing
coverage can influence apparent changes in the spatial mean. No gaps are interpolated.

In [ ]:
series = pd.DataFrame({
    "Ground snow fraction (%)": scfg.mean(["lat", "lon"]).to_series(),
    "Snow water equivalent (mm)": swe.mean(["lat", "lon"]).to_series(),
})
coverage = pd.DataFrame({
    "SCFG": scfg.notnull().mean(["lat", "lon"]).to_series(),
    "SWE": swe.notnull().mean(["lat", "lon"]).to_series(),
})
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 8))
series.iloc[:, 0].plot(ax=axes[0], ylabel="Snow fraction (%)", ylim=(0, 100))
series.iloc[:, 1].plot(ax=axes[1], ylabel="SWE (mm)")
coverage.plot(ax=axes[2], ylabel="Observed cell fraction", ylim=(0, 1))
fig.tight_layout()
plt.show()
display(series.describe())

## Snow extent on one date

The CLMS Northern Hemisphere snow-extent field and the CCI ground-fraction field are shown for
15 May 2020 over the same geographic window. Both are delivered in percent, but they remain
separate products with different grids and processing. They are displayed independently without
resampling one into the other's grid or treating their difference as a validated bias estimate.

The CLMS SCE delivery retains digital codes 100–200. Physical snow cover is `code - 100`;
other codes represent masks and are excluded. This conversion follows the
[Copernicus product specification](https://land.copernicus.eu/en/products/snow/snow-cover-extent-northern-hemisphere-v1-0-1km)
and the [EO4FLOOD product specification](https://eo4flood.org/wp-content/uploads/2026/03/EO4FLOOD_D2.3_PSD_v3.pdf).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for field, label, ax in zip([scfg.sel(time="2020-05-15"), sce],
                            ["CCI ground snow fraction", "CLMS snow cover extent"], axes):
    field.plot(ax=ax, x="lon", y="lat", cmap="Blues", vmin=0, vmax=100,
               cbar_kwargs={"label": "Percent"})
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.set_title(label)
fig.tight_layout()
plt.show()
for dataset in datasets.values():
    dataset.close()